# xgap: demo replay (step 2)

Thin by design -- no logic lives here. Mount Drive, clone/pull the repo,
run `setup_colab.sh`, call `scripts/run_demo_replay.py`. All control flow
lives in `xgap_code/` and `scripts/`; this notebook only sequences calls to
it.

Repo: https://github.com/AITEAM444/xgap (public)

**Run cell 1 before importing anything else, in every fresh runtime.** Colab's
`!` shell subprocess env does not propagate into this kernel's own Python
process, so `setup_colab.sh` cannot set `MUJOCO_GL` for you here -- see that
script's own comments.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

XGAP_DRIVE_ROOT = "/content/drive/MyDrive/xgap"
XGAP_REPO_URL = "https://github.com/AITEAM444/xgap.git"

Mounted at /content/drive


## Get the code

Clones into Drive on first run, `git pull`s on every run after -- code/config
history stays on GitHub (the source of truth), Drive is just where it lives so
`setup_colab.sh` / scripts can read it and outputs can be written next to it.
No manual re-uploading of files to Drive after this point; push to GitHub and
re-run this cell instead.

In [ ]:
os.makedirs(XGAP_DRIVE_ROOT, exist_ok=True)

if os.path.isdir(f"{XGAP_DRIVE_ROOT}/.git"):
    # Already a clone -- re-point + fast-forward rather than a bare `git pull`.
    !git -C {XGAP_DRIVE_ROOT} remote set-url origin {XGAP_REPO_URL}
    !git -C {XGAP_DRIVE_ROOT} fetch origin
    !git -C {XGAP_DRIVE_ROOT} checkout -B master origin/master
else:
    # No `.git` here, but `git clone` refuses to clone into a non-empty directory --
    # and this directory is never actually empty in practice, since setup_colab.sh
    # points HF_HOME at a `.hf_cache/` subfolder of it. `git init` + `fetch` + `checkout`
    # only touches files git itself tracks, so it works in-place regardless of what
    # other untracked stuff (like `.hf_cache/`) already lives here.
    !git -C {XGAP_DRIVE_ROOT} init
    !git -C {XGAP_DRIVE_ROOT} remote add origin {XGAP_REPO_URL}
    !git -C {XGAP_DRIVE_ROOT} fetch origin
    !git -C {XGAP_DRIVE_ROOT} checkout -B master origin/master

Branch 'master' set up to track remote branch 'master' from 'origin'.
Reset branch 'master'
Your branch is up to date with 'origin/master'.


## Environment rebuild

Idempotent -- safe to re-run. This also appends `nproc` / `nvidia-smi` /
`free -g` / installed library versions to `logs/env_meta.log` (Colab hardware
varies session to session), which satisfies the "log hardware in the first
cell" requirement without duplicating that logic here.

If this prints a restart banner, use *Runtime > Restart session* and re-run
this cell once (it will no-op on the already-satisfied install step) before
continuing.

In [ ]:
!bash {XGAP_DRIVE_ROOT}/setup_colab.sh

[xgap setup] XGAP_DRIVE_ROOT=/content/drive/MyDrive/xgap
[xgap setup] HF_HOME=/content/drive/MyDrive/xgap/.hf_cache
[xgap setup] HF_LEROBOT_HOME=/content/xgap_hf_lerobot_cache
[xgap setup] LIBERO_CONFIG_PATH=/content/xgap_libero_config
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.7/217.7 kB 25.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.9/192.9 kB 23.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 20.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.5/217.5 kB 27.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 56.3

## Adopt resolved env vars into this kernel

`setup_colab.sh`'s own `export`s (`HF_HOME`, `HF_LEROBOT_HOME`,
`LIBERO_CONFIG_PATH`) only apply to ITS OWN subprocesses -- they vanish once
that script exits, same as any other `!`-cell `export`. Without this cell,
the next cell (a separate `!python ...` subprocess) sees none of them: caches
silently fall back to their unconfigured defaults, and `libero`'s interactive
dataset-path prompt reappears even though it already passed inside
`setup_colab.sh`. Reads `/content/.xgap_env` (written by the script above) into
`os.environ` in the KERNEL process instead, which every `!` cell for the rest
of this session DOES inherit -- same mechanism as why cell 1's `MUJOCO_GL`
works.

In [ ]:
with open("/content/.xgap_env") as f:
    for line in f:
        key, _, value = line.strip().partition("=")
        if key:
            os.environ[key] = value

for _k in ["XGAP_DRIVE_ROOT", "MUJOCO_GL", "HF_HOME", "HF_LEROBOT_HOME", "LIBERO_CONFIG_PATH"]:
    print(f"{_k}={os.environ.get(_k)}")

XGAP_DRIVE_ROOT=/content/drive/MyDrive/xgap
MUJOCO_GL=egl
HF_HOME=/content/drive/MyDrive/xgap/.hf_cache
HF_LEROBOT_HOME=/content/xgap_hf_lerobot_cache
LIBERO_CONFIG_PATH=/content/xgap_libero_config


## Smoke run first

`configs/demo_replay_smoke.yaml`: 1 suite, 1 task, up to 5 episodes, both
`control_mode`s, with per-episode `.mp4` + trajectory plots saved locally
(`outputs/demo_replay_smoke/_local/videos/`) for visual inspection. See the
repo README, "How to read the first real run", for how to tell a crash apart
from an actual low-success-rate finding.

If re-running after a code change to the replay/logging/video pipeline
itself (not just a config change), clear old results first -- resume only
checks "does this episode's result file already exist", not whether it has
the fields/videos the current code would produce:

```python
!rm -rf /content/outputs/demo_replay_smoke
!rm -rf /content/drive/MyDrive/xgap/outputs/demo_replay_smoke
```

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay_smoke.yaml

## control_freq comparison (done -- kept for reference)

`configs/demo_replay_smoke_cf20.yaml` -- same task/episode/control_modes as
the smoke config above, `control_freq=20` instead of `10`. This was run:
`control_freq=20` is now confirmed correct (see README "control_freq was
wrong from the start" -- LIBERO's own source shows demos are collected and
replayed 1:1 at 20Hz; `demo_replay_smoke.yaml`'s default was corrected to
match) and gets the eef position trajectory to track the recorded demo
almost exactly. But grasping still fails at `control_freq=20` too --
`gripper_qpos` closes fully (nothing between the fingers) both times, where
the recorded demo shows a partial close (something between the fingers)
both times. So this cell is no longer diagnostic on its own -- kept so the
comparison is reproducible -- and the investigation moved to the two cells
below instead.</cell id="b92b2689">


In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay_smoke_cf20.yaml

## Sanity check against a KNOWN answer -- DONE, infrastructure confirmed sound

None of this project's own code runs in this cell -- just `lerobot`'s own
`lerobot-eval` CLI against `lerobot/pi05_libero_finetuned`, a checkpoint with
a published official number: **96% on LIBERO-10** (see
`docs/source/libero.mdx` in lerobot, "Reproducing published results"). No
50GB dataset download needed either -- eval only touches the environment and
the (small) policy checkpoint.

**Result:** every task that finished before the run was stopped hit
`running_success_rate=100.0%` on 5/5 episodes (20/20 episodes across 4
completed tasks). See README "Sanity check result: infrastructure is
confirmed sound, the bug is ours" for the full story, including two false
alarms along the way worth knowing about before reading this cell's output:
`Stepping through eval batches: N/5` is a *per-task* bar that resets to a
fresh 0% every time a new task starts (looked like an ongoing failure mid-run,
was actually just a fresh counter), and the piped-to-file tqdm log is huge
(one line per step) so skimming only the tail can land inside an
in-progress episode that structurally shows 0.0% until it finishes --
grep the file for `100%` to find actual per-task completions instead.

**Fixed below:** `--env.task_ids=[0]` now restricts to a single task, since
`--env.task=libero_10` alone evaluates all 10 tasks in the suite (5 episodes
*per task*, not 5 total as originally intended) -- this is what actually
made the first run take ~50 episodes instead of the quick 5-episode check
this cell was meant to be.

**Needs a GPU runtime** (Runtime > Change runtime type > GPU) -- CPU-only
inference of a VLA policy is slow enough to look hung.

**Gated dependency:** this checkpoint's VLM backbone needs
`google/paligemma-3b-pt-224`, which Google gates. One-time setup: accept
the license at https://huggingface.co/google/paligemma-3b-pt-224 while
logged into your HF account, then in a cell: `from huggingface_hub import
login; login(token="...")` (token from
https://huggingface.co/settings/tokens).

**Output redirected to a log file, not printed live** -- `lerobot-eval`'s
per-step logging is verbose enough that letting Colab render it live can
overload the browser tab and disconnect the runtime (hit this once). The
cell below writes everything to `/content/lerobot_eval.log` and only prints
the last 80 lines; `!cat /content/lerobot_eval.log` in a scratch cell if you
need the full log, and `grep -c '100%' /content/lerobot_eval.log` to count
completed-task markers without reading the whole thing.</cell id="bb029484">


In [ ]:
!lerobot-eval \
    --policy.path=lerobot/pi05_libero_finetuned \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval.log 2>&1
!tail -n 80 /content/lerobot_eval.log

## SmolVLA directly via `lerobot-eval` -- no xgap code

Same idea as the sanity-check cell above, but for the actual checkpoint
under test (`HuggingFaceVLA/smolvla_libero`) instead of the known-good
`pi05` reference. This is the direct answer key for `xgap`'s own N=1 result
-- and doubles as the Gate-1 baseline -- without needing the demo-replay
init_state mapping at all (see README "Mapping search abandoned": policy
eval never uses a demo's initial state, only `init_states` in plain order,
which is exactly what this runs).

No `--policy.n_action_steps` override -- let the checkpoint's own config
supply it rather than reusing `pi05`'s `10` by copy-paste. Same GPU
requirement and log-redirect reasoning as the cell above.

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval_smolvla.log 2>&1
!tail -n 80 /content/lerobot_eval_smolvla.log

**Result: 0/5, confirmed via pure `lerobot-eval` -- checkpoint-specific, not
an xgap harness bug.** See README "Mapping search abandoned" section for the
full writeup. Wrist-camera input (H2) is not a live suspect for this result
(`lerobot-eval`'s own env supplies both cameras automatically). Next: H1 --
sweep `n_action_steps` one value at a time, still via pure `lerobot-eval`.

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval_smolvla_nas10.log 2>&1
!tail -n 80 /content/lerobot_eval_smolvla_nas10.log

**Result: 0/5 again -- H1 (execution granularity) rejected, 1 vs 10 makes
no difference.** `eval_ep_s` dropped 462s -> 145.6s (~3.2x) on a GPU
upgrade (T4->A100) *and* 10x fewer policy calls combined -- much less than
compute-bound scaling would predict, so the real bottleneck is very likely
env stepping/rendering, not policy inference (see README).

Two higher-information, untested axes come next instead of continuing this
sweep to 25/50 (which would only add a third "0/5, learned nothing new"
point) -- see README "Higher-priority than finishing the `n_action_steps`
sweep": (1) does this checkpoint know `libero_10` at all (try
`libero_spatial` instead), (2) the unresolved 360-vs-256 resolution
question from Step 1.

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_spatial \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval_smolvla_spatial_nas10.log 2>&1
!tail -n 80 /content/lerobot_eval_smolvla_spatial_nas10.log

**Result: 3/5 (60%) -- decisive.** Checkpoint/policy/env/`lerobot-eval` loop
all genuinely work; the failure is specific to `libero_10`, not SmolVLA in
general. Also weakens (doesn't kill) the resolution hypothesis below: this
ran at the same default 360 resolution as every failing `libero_10` run and
still succeeded 60% of the time. See README for the full writeup and the
two remaining candidate explanations (task difficulty vs something
`libero_10`-scene-specific).

## Resolution mismatch check (unresolved since Step 1)

Env default render is 360, checkpoint declares 256, and there's a separate
512-padding setting on top -- never confirmed from source whether/where a
resize actually reconciles these. Check the real CLI field names first
(cheap, no GPU) so the eval below doesn't run on a typo'd flag:

In [ ]:
import dataclasses
from lerobot.envs.configs import LiberoEnv
print([f.name for f in dataclasses.fields(LiberoEnv)])

**Confirmed:** `observation_height`/`observation_width` are real fields
(full list: `task`, `fps`, `features`, `features_map`, `max_parallel_tasks`,
`disable_env_checker`, `task_ids`, `episode_length`, `obs_type`,
`render_mode`, `camera_name`, `init_states`, `camera_name_mapping`,
`observation_height`, `observation_width`, `is_libero_plus`,
`control_mode`) -- the eval cell below uses the right flag names.

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --env.observation_height=256 \
    --env.observation_width=256 \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval_smolvla_res256.log 2>&1
!tail -n 80 /content/lerobot_eval_smolvla_res256.log

**Result: 1/5 (20%), episode 3 only -- weak, inconclusive at n=5.** Not a
clean confirm/reject like `libero_spatial`. Noted as unresolved, not
pursued further with more episodes for now (see README).

## Gripper-close duration, not just pass/fail

Three 0/5s in a row say nothing new on their own -- but whether the longest
continuous closed-gripper run (`xgap_code/gripper_metrics.longest_close_run`;
a real demo grasp holds closed 15-20+ consecutive steps, measured earlier
from real demo data) moves between conditions splits "never attempts to
close" from "closes but doesn't grasp" -- different bugs. Check whether
plain `lerobot-eval` already saves raw per-step actions anywhere before
assuming a new script is needed:

In [ ]:
!ls -la outputs/eval/*/*/
!python -c "import lerobot.scripts.eval as e; print(e.__file__)"

**Result: `eval_info.json` is just the same aggregated summary already
printed to stdout -- no raw actions.** Module-path guess was also wrong;
real one (from `cat $(which lerobot-eval)`) is
`lerobot.scripts.lerobot_eval`. Reading that module's actual source paid
off: `eval_policy()`/`rollout()` already have a full recording path that
writes a real `LeRobotDataset` (raw `action` per frame) to
`<output_dir>/recordings/<task_group>_<task_id>/` -- `eval_main()` just
never turns it on by default (`cfg.eval.recording=False`). No new rollout
script needed -- just the flag, plus a small reader
(`scripts/gripper_close_from_recording.py`) for the resulting local
parquet, reusing `gripper_metrics.longest_close_run` directly. See README
for the full writeup (including the videos-off tradeoff when recording is
on).

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --eval.recording=true \
    --env.max_parallel_tasks=1 \
    --output_dir=/content/outputs/eval_gripper_libero10 \
    > /content/lerobot_eval_smolvla_gripper_libero10.log 2>&1
!tail -n 40 /content/lerobot_eval_smolvla_gripper_libero10.log
!python {XGAP_DRIVE_ROOT}/scripts/gripper_close_from_recording.py \
    --recording-dir /content/outputs/eval_gripper_libero10/recordings/libero_10_0

## Critical check: is the policy's actual input image mirrored?

Raised after watching the baseline videos: object text ("Milk", "Orange
Juice") in `libero_10` frames reads as a mirror image. The saved `.mp4`
(`env.render()`) and the tensor actually fed to `policy.select_action()`
(`preprocess_observation()` -> `env_preprocessor()` -> `preprocessor()`)
are DIFFERENT code paths -- a flip in one doesn't imply a flip in the
other, so this must be checked on the real policy-input path directly, not
inferred from video. See README "Open thread that could overturn the
verdict above" for the full reasoning, including why `pi05`'s 20/20 on
this exact setup makes a policy-input mirror unlikely but not yet ruled
out -- if this comes back mirrored, the checkpoint-weakness verdict above
does not hold.

`scripts/check_image_mirroring.py` builds the observation the same way
`lerobot-eval`'s own `rollout()` does internally (same public functions,
same order, no monkey-patching) for one `env.reset()`, dumps both cameras'
actual policy-input tensors as PNG, and fetches one same-task reference
frame directly from `HuggingFaceVLA/libero` to compare against.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/check_image_mirroring.py \
    --out-dir /content/drive/MyDrive/xgap/outputs/mirror_check

**Result: not mirrored.** Both cameras' policy-input PNGs match the
dataset-reference PNGs' left-right orientation exactly. The mirrored text
was specific to the saved `.mp4` rendering path, not the policy's actual
input. Closes the last thread that could have overturned the
checkpoint-weakness verdict -- it stands. See README for the full
writeup.

## init_state sweep (is `within_task_index` picking the wrong index?)

`configs/sweep_init_states.yaml`: fixes ONE demo's recorded actions
(`libero_10` task 0, dataset episode_index=8 -- the same episode used in the
comparisons above) and replays them against **every** candidate `init_state`
LIBERO has for this task (`scripts/sweep_init_states.py`, no new
instrumentation -- one loop over `harness.get_num_init_states()`). Cheap
relative to another position/orientation investigation: same ~300-step
episode run N times (N = however many init_states this task has, LIBERO's
own count, not assumed), no additional download.

- **Any index succeeds** -> `within_task_index` (`dataset_io.py`)
  is picking the wrong LIBERO init_state, confirmed, and this sweep hands
  you the *correct* index directly (`sweep_summary.json`'s
  `successful_init_states`).
- **None succeed** -> init-state indexing is exonerated. Orientation
  (`eef_quat`, not currently in `state_chunk`) becomes the next thing to
  add and check -- see README.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/sweep_init_states.py \
    --config {XGAP_DRIVE_ROOT}/configs/sweep_init_states.yaml

## Visually verify one successful init_state

`scripts/sweep_init_states.py`'s resume logic keys on "does this episode's
result already exist" -- turning video on for an index it already ran would
just skip it silently and never produce a video. This is a separate,
always-re-runs script for spot-checking any ONE index from the sweep's
`successful_init_states` with an actual `.mp4` + trajectory-vs-demo overlay
plot, without touching the sweep's own results. Swap `--init-state-index`
for whichever index you want to look at.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/render_init_state_video.py \
    --config {XGAP_DRIVE_ROOT}/configs/sweep_init_states.yaml \
    --init-state-index 3 \
    --out-dir /content/outputs/init_state_render

## Full demo replay

Only run this after the smoke cell above completes cleanly (prints a
`decision` JSON, not a traceback).

## Pivot: libero_spatial replaces libero_10 as the Gate-1 target

See README "Decision: the August gate moves to LIBERO-Spatial" for the
full reasoning. In short: the proposal's own criterion for "saturated"
(judged against the policy actually being used, not SOTA) puts
`libero_spatial` in the same open-headroom band as the small-model
literature it already cited (53.7-64.8%) -- SmolVLA's real 60% there fits
that band and clears Gate 1. `libero_10` has no working baseline to
measure anything against (0%), so there is nothing to apply pressure to.
`libero_10` is not deleted -- it's carried forward as an extension
experiment, to revisit after additional fine-tuning if H-axis headroom is
needed later.

Before scaling up, one parity check: does `xgap`'s OWN harness
(`harness.make_real_libero_env`, standard init_state order -- see that
function's docstring) reproduce the same real-world result as the official
`lerobot-eval` CLI's `libero_spatial` task 0 result (3/5, 60%,
`n_action_steps=10`, above)? This validates the harness path the actual
Gate-1/N/H experiments will need (for `n_decision_points` /
`exec_horizon` / `selection_unit` control `lerobot-eval`'s CLI doesn't
expose), before trusting numbers from it. `scripts/run_policy_rollout.py`
+ `xgap_code/policy_rollout.py` -- reuses lerobot's own policy-loading and
observation-preprocessing pipeline directly (same functions
`check_image_mirroring.py` already proved work end to end in this exact
Colab environment), stepping through with `policy.select_action()`
per-step (matches the project's own confirmed n_action_steps=1-vs-10
null result, so a chunked executor isn't needed for this check) against
xgap's own env instead of lerobot's. One known unverified risk, flagged
in the module docstring rather than hidden: lerobot's preprocessing
pipeline expects a BATCHED observation (from its own vectorized env);
xgap's harness env is unbatched, so this manually adds/removes a batch
dimension of 1 to bridge that -- watch for a shape-mismatch error here
first if this run fails.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_policy_rollout.py \
    --config {XGAP_DRIVE_ROOT}/configs/policy_rollout_libero_spatial_smoke.yaml

**Result: 2/5 (40%) via xgap's own harness vs 3/5 (60%) via `lerobot-eval`
-- one episode apart, at n=5.** Not identical, but n=5 is too small a
sample to distinguish "harness bug" from ordinary noise (a single episode
flip moves this rate by 20pp either way), and this checkpoint likely has
some inherent run-to-run stochasticity in its own action sampling on top
of that. Both successes finished quickly (77, 73 steps) with a plausible
`longest_close_run` (36-39); all three failures ran the full 520-step cap
rather than ending early -- an ordinary failure shape, not an obviously
broken one.

**Decision (see README "cost-aware sizing"): don't spend a separate
session getting more confidence in this comparison specifically.** The
real Gate-1 measurement (3-5 tasks x 15-20 episodes) will include ~15-20
episodes on this same task 0 anyway -- that larger sample IS the harness
parity check, at higher n, for free. Running a bigger n=15 parity check
first would only buy "the harness is probably right" without producing
the Gate-1 number itself; reordering to go straight to the full
measurement gets both from one run.

## Gate-1 measurement, sized by cost

Real timing from the smoke run: `avg_success_episode_s=89.1`,
`avg_fail_episode_s=493.6` (~5.5x, confirming failed episodes running the
full `max_steps` cap dominate cost). At this checkpoint's 40-60% failure
rate on `libero_spatial`, expected cost is ~4.2-5.5 min/episode.

**Decision: 3 tasks x 20 episodes = 60 episodes (~4.2-5.5 hours).**
`configs/policy_rollout_libero_spatial_gate1.yaml` -- task_ids `[0, 1, 2]`.
Task 0's own 20-episode subset also serves as the harness-parity check
against `lerobot-eval`'s 3/5 reference, at 4x the smoke run's n (see
README "Cost-aware sizing"). Episode-level incremental save + resume
(`EpisodeStore`, already built for exactly this) means a Colab disconnect
mid-run loses at most the one in-flight episode, not the whole run -- just
re-run the same cell to continue.

In [ ]:
!pip install num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 15.6 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=58b2d35156cdf143ab995773b428202882f53840ed2e428db46dd6234e1899fd
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_policy_rollout.py \
    --config {XGAP_DRIVE_ROOT}/configs/policy_rollout_libero_spatial_gate1.yaml

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[rollout] task='pick up the black bowl between the plate and the ramekin and place it on the plate' (libero_spatial:0)
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 489/489 [00:46<00:00, 10.41it/s]
  task=0 episode=0: success=True (resumed)
  task=0 episode=1: success=True (resumed)
  task=0 episode=2: success=False (resumed)
  task=0 episode=3: success=True (resumed)
  task=0 episode=4: success=True (resumed)
  task=0 episode=5: success=True (resumed)
  task=0 episode=6: success=True (resumed)
  task=0 episode=7: success=False (resumed)
  task=0 episode

## Test 2: state save/restore determinism (parallel track)

Does "same state + same action = same result" actually hold? This
underlies every future Oracle/World-model/Random candidate comparison
(see README "Design constraints") -- if restoring a saved simulator state
and replaying the same action doesn't reproduce the same outcome,
comparing candidates branched from a shared decision point is meaningless.
Flagged in this project's own planning as the hardest implementation
element, untouched until now -- can run independently of (and at the same
time as) the Gate-1 measurement above.

`scripts/test_state_restore_determinism.py`: per trial, reset + a few
warmup steps, save state (`harness.get_sim_state`,
`env.sim.get_state().flatten()` -- the standard robosuite/MuJoCo idiom,
same method name already documented in `dataset_io.py`'s noted-but-not-
implemented fallback plan, but its first real use in this project), branch
A steps one fixed test action, restore the saved state
(`harness.restore_sim_state`), branch B steps the SAME action again,
compare A vs B (state + rendered image).

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/test_state_restore_determinism.py \
    --task-suite libero_spatial --task-id 0 --n-trials 5

**Result: NOT bit-identical, but small and consistent.** All 5 trials: state
max_abs_diff ~0.0014-0.0016 (tight clustering, not random-magnitude noise),
image ~0.18-0.22% of pixels differing, **reward identical in all 5**. Most
likely explanation: mujoco_py's `MjSimState` (time, qpos, qvel, act) does
not include the contact solver's warm-start acceleration
(`qacc_warmstart`) -- physics is deterministic given the TRUE internal
solver state, but this flattened snapshot may not capture all of it.

Doesn't answer the question that actually matters for candidate
comparison: does this small per-step gap compound, stay flat, or shrink
over an `exec_horizon`-scale rollout (not just 1 step)? Extended below.

## Test 2, revised: signal-vs-noise, not absolute magnitude

The single-step result's reward-matched-5/5 isn't reassuring on its own --
LIBERO's success signal is binary/coarse (a few mm of object displacement
doesn't register), so it can't detect the residual either way. The only
metric that actually matters: is restore-noise small RELATIVE TO the
state divergence a genuinely different candidate action would produce?
`test_state_restore_determinism.py` now measures both directly from the
same saved state -- **signal**: two DIFFERENT fixed action chunks run for
`--n-compare-steps` each (as if two different policy/world-model
proposals); **noise**: the SAME chunk run twice (once, then again after a
restore). Reports `signal/noise` ratio per step. Threshold this project is
using: ~100x is practically safe, ~10x is risky, ~1x means candidate
comparison doesn't work.

**Run this BEFORE the Gate-1 measurement above, not in parallel with it**
-- both need the GPU, and this is the cheaper, more urgent question: if
the ratio is bad, the whole premise of any future candidate comparison
(and possibly this harness's own success/failure precision near a
decision boundary) needs fixing before a multi-hour run is worth trusting.
Stop the Gate-1 cell first if it's already running -- `resume: true` means
nothing already completed is lost, re-running the same cell later
continues rather than restarts.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/test_state_restore_determinism.py \
    --task-suite libero_spatial --task-id 0 --n-trials 3 --n-compare-steps 10

**Result: FAIL. min ratio = 1.0x (avg ratio at last step 51.0x).** Seed 0's
noise matched signal at every one of 10 compared steps (never recovered).
Seeds 1/2 also started at ~1.0x for the first several steps, then noise
decayed roughly exponentially while signal stayed roughly constant,
pulling the ratio up to 16-124x by the last step -- consistent with the
contact solver re-converging over a few steps from a missing warm-start
(`qacc_warmstart` isn't part of `MjSimState`), not from missing
qpos/qvel/time/act (which *are* restored). Seed 0 never recovering
suggests that restore may have landed in a genuinely different contact
configuration, not just a transient convergence artifact.

**Fix applied, not yet verified:** `harness.get_sim_state`/
`restore_sim_state` now also capture/restore `sim.data.qacc_warmstart`
directly (a plain array attribute, independent of `MjSimState`) alongside
the flattened state. Re-run the same command above -- if this was the
cause, the ratio should improve especially at the early steps that were
previously stuck at ~1.0x.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/test_state_restore_determinism.py \
    --task-suite libero_spatial --task-id 0 --n-trials 3 --n-compare-steps 10

**`qacc_warmstart` fix tested and REJECTED.** Confirmed captured
(`shape=(43,)`, 39/43 nonzero -- a real buffer, via a diagnostic print
added specifically for this) and restored every trial, but the
signal/noise arrays came back bit-for-bit identical to the pre-fix run.
Makes sense in hindsight -- warm-starting an iterative solver should only
affect convergence speed, not the converged result.

**Revised hypothesis: the robot CONTROLLER's own Python-level state, not
physics.** `restore_sim_state` only ever touched `env._env.sim` -- the
OSC_POSE controller (`env._env.robots[0].controller`) is a separate
Python object with its own array/scalar memory (integral terms,
previous-command memory, ramp/filter buffers) entirely outside `sim`. A
physics-only restore leaves that memory stale -- matches the observed
pattern exactly (large noise right after restore, decaying over several
steps as new commands overwrite it). This is the exact gap
`restore_sim_state`'s docstring flagged from the start.

**Fix applied, not yet verified:** `harness.get_sim_state`/
`restore_sim_state` now also snapshot/restore the controller's
array/scalar attributes (skipping object references, so this never risks
deep-copying the whole simulator). A second diagnostic print reports
whether `robots[0].controller` was found and which keys got captured.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/test_state_restore_determinism.py \
    --task-suite libero_spatial --task-id 0 --n-trials 3 --n-compare-steps 10

**Result: no meaningful change, confirmed with controller state actually
captured** (`goal_pos`, `goal_ori`, `joint_pos`, `mass_matrix`, etc. --
the full array/scalar set). Checked the interpolator hypothesis directly:
`controller.interpolator_pos`/`interpolator_ori` are both `None` -- not in
use, nothing hidden there. Every Python-level state component found so
far (physics, `qacc_warmstart`, full controller memory) is now restored,
residual unchanged. Remaining candidate is MuJoCo/robosuite API-level
(`sim.forward()` possibly not fully reconstructing what a normal `step()`
sequence leaves behind) -- would take real time to root-cause with no
guaranteed fix.

## Test 2: design change to prefix replay

**Not an unresolved failure -- a design change.** State save/restore was
never the goal, only one possible *means* to it; the actual goal is fair
candidate comparison from a shared decision point, and there's a second
way to get there. Four weeks out from the gate (2026-08-03), continuing
to root-cause an API-level MuJoCo limitation that might still end in "no
fix exists" isn't worth it.

**Alternative: prefix replay, not restore.** `env.reset(seed)` -> replay
a recorded prefix action sequence to the branch point -> run ONE
candidate's action(s) -> repeat per candidate, same seed + same prefix
each time. No state is ever saved or restored -- MuJoCo is deterministic
by construction given the same seed + action sequence, so the branch-point
state is exactly what a normal `step()` sequence produces, because it IS
one. Whatever `sim.forward()` couldn't reconstruct is simply never at
issue. Cost: ~2x a single rollout at a mid-episode branch point (the
shared prefix gets replayed per candidate) -- worse than save/restore's
O(1) would have been, but budgeted, not hidden (a shorter timeout, e.g.
300 vs 520 steps, partially offsets it).

Verification is symmetric with the original test: run the SAME `(seed,
prefix, post-branch action)` through `reset()+step()` TWICE and require
**exactly bit-identical**, not merely small -- MuJoCo's physics is
deterministic, so a real match should be exact; anything that only
shrinks toward zero signals a different, still-open nondeterminism source.

Does not block the Gate-1 measurement above -- Gate-1 never used
save/restore (single, unbranched rollout per episode). Start it now if
not already running; this verification can run independently afterward.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/test_prefix_replay_determinism.py \
    --task-suite libero_spatial --task-id 0 --n-trials 3 --prefix-steps 10 --post-branch-steps 10

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Local assets not found. Downloading from HuggingFace Hub...
Assets already downloaded at /root/.cache/libero/assets
seed=0 n_steps=21 all_exact_zero=True
  reset_diff (index 0, before any action)=0.0 exact_zero=True
  diff_per_step (index 0 = reset, 1..N = post-action)=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
seed=1 n_steps=21 all_exact_zero=True
  reset_diff (index 0, before any action)=0.0 exact_zero=True
  diff_per_step (index 0 = reset, 1..N = post-action)=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
seed=2 n_steps=21 all_exact_z

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay.yaml

In [ ]:
import inspect
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
print([m for m in dir(SmolVLAPolicy) if not m.startswith('_') or m in ('_get_action_chunk', '_queues')])
print("---select_action---")
print(inspect.getsource(SmolVLAPolicy.select_action))


## Gate 2: candidate diversity

Does the policy propose meaningfully different candidates from the same
observation, or does it collapse to a single mode? Raised after task 1's
Gate-1 run showed 19/20 episodes with near-identical `rollout_length`
(95-121) and `longest_close_run_steps` (39-63). Checked on the actual
experiment-set tasks (0, 2, 4, 5 -- 1 deliberately excluded, it's the one
that already showed the concern). See README "Gate 2: candidate
diversity" for the full design (why `predict_action_chunk(noise=None)` is
confirmed from source to give independent stochastic samples, why branch
points come from real recorded Gate-1 episodes via prefix replay, and the
cost tradeoff behind starting at `branch_step_fractions: [0.2]`).

Requires task 0's and task 2's episode 0 (from the main Gate-1 config) and
task 4's/task 5's episode 0 (from the per-teammate configs) to already
exist under the shared `policy_rollout_libero_spatial_gate1` output --
will error clearly if a source episode is missing rather than silently
skip it.

This script reports raw numbers, not an automatic verdict -- "meaningfully
different" vs "nearly identical" needs a judgment call against real
physical scale, see printed `per_dimension_std` / `mean_pairwise_l2` /
`gripper_distribution` / `endpoint_variance` per task.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_gate2_diversity.py \
    --config {XGAP_DRIVE_ROOT}/configs/gate2_diversity.yaml

## Gate 2, real results: outcome-based redesign (finalized)

**Real results (task 0, task 2, `branch_step_fractions: [0.2]`):** genuine
action-level diversity (`mean_pairwise_l2` ~4.2-4.6, gripper channel split
~50/50 across 64 candidates) but `endpoint_variance.total_variance` came
back ~300x smaller in proportion (~0.017 / ~0.0136), and rotation-axis
endpoint variance was ~1e-11 -- effectively deterministic. Candidates issue
meaningfully different commands but arrive almost at the same place.

**Resolved, not just ambiguous: the endpoint-variance metric is
INVALIDATED for verdict purposes.** `gripper_channel_distribution`'s
`frac_commanding_close` went 0.53 at `branch_fraction=0.2` to 0.85 at
0.5 -- candidates really DO differ substantially on WHEN they command the
gripper closed. But `exec_horizon=10` is shorter than the gripper's own
15-20 step actuation lag (`gripper_metrics.DEMO_MIN_ACTUATION_LAG_STEPS`),
so that real command-timing difference has no physical window in which to
become an endpoint difference. See README "Gate 2: real results, and
redesign to outcome-based verdict" for the full reasoning.

**Endpoint-variance measurement stops once the 0.7 point above finishes
running.** No further fractions to add; its result from here on is kept
only as diagnostic evidence, not used to pick anything.

**Gate 2's verdict now comes from episode OUTCOME alone --
`branch_fraction=0.5`, FIXED by this decision.** See README "Gate 2,
corrected: outcome must branch from failures too, not only successes" --
the first attempt at this outcome script had two design bugs, both fixed
before any real run consumed the budget:

1. Only 5 trials per point isn't enough evidence to conclude "no
   diversity" from an all-same result -- now `n_candidates=64`
   (early-stopping once a mix IS observed is still fine, only the
   all-same ceiling changed).
2. Branching only from an already-SUCCESSFUL source episode
   near-guarantees the handoff also succeeds regardless of candidate
   choice, which tests nothing about Gate 2 -- `source_episode_seeds` is
   now a dict per task mixing real recorded SUCCESS and FAILURE episodes,
   with failure the more important case ("was about to fail, a good
   candidate flips it to success" is literally the Oracle-curve
   mechanism).

**Before running the outcome cell, find real seeds per task** (next cell
below) -- `configs/gate2_outcome.yaml` ships with placeholder seeds
`[0, 1, 2, 3]` per task that must be replaced with real success/failure
seeds first.

In [ ]:
for _task_id in [0, 2, 4, 5]:
    print(f"=== task {_task_id} ===")
    !python {XGAP_DRIVE_ROOT}/scripts/list_source_episode_outcomes.py \
        --source-output-root /content/drive/MyDrive/xgap/outputs/policy_rollout_libero_spatial_gate1 \
        --condition policy_rollout --task-suite libero_spatial --task-id {_task_id}

**Now edit `configs/gate2_outcome.yaml`'s `source_episode_seeds`** using the
real success/failure seeds printed above -- pick ~2 success + ~2 failure
per task (failure matters more, see above), replacing the placeholder
`[0, 1, 2, 3]` lists. Commit + push + re-run the "Get the code" cell (or
edit the file directly on Drive) before running the cell below.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_gate2_outcome.py \
    --config {XGAP_DRIVE_ROOT}/configs/gate2_outcome.yaml